Checking the rss feeds

In [ ]:
import feedparser

feeds = {
    "bbc_sport":"http://feeds.bbci.co.uk/sport/rss.xml",
        "epsn_sport":"https://www.espn.com/espn/rss/news",
        "sky_sport":"https://www.skysports.com/rss/12040",
        "epsn_cric":"https://www.espncricinfo.com/rss/content/story/feeds/0.xml",
        "f1_sport":"https://www.motorsport.com/rss/f1/news/",

        "the_hindu":"https://www.thehindu.com/feeder/default/rss",
        "scmp":"https://www.scmp.com/rss/91/feed",
        "japan_times":"https://www.japantimes.co.jp/feed/topstories/",
        "nikkei_asia":"https://asia.nikkei.com/rss/feed/nar",
        "times_of_india_world":"https://timesofindia.indiatimes.com/rssfeeds/296589292.cms",
        "hindustan_times":"https://www.hindustantimes.com/feeds/rss/world-news/rssfeed.xml",
    

        "aljazeera": "https://www.aljazeera.com/xml/rss/all.xml",
        "times_of_israel": "https://www.timesofisrael.com/feed/",
        "egypt_independent": "https://egyptindependent.com/feed/",
    "middle_east_eye": "https://www.middleeasteye.net/rss",

        "all_africa": "https://allafrica.com/tools/headlines/rdf/latest/headlines.rdf",
        "news24_za":      "https://feeds.news24.com/articles/news24/TopStories/rss",
        "mail_guardian":  "https://mg.co.za/feed/",
        "premium_times":  "https://www.premiumtimesng.com/feed",


        "rte": "https://www.rte.ie/feeds/rss/?index=/news",
        "bbc": "https://feeds.bbci.co.uk/news/rss.xml",
        "dw": "https://rss.dw.com/rdf/rss-en-all",
        "le_monde": "https://www.lemonde.fr/en/rss/une.xml",
        "france24": "https://www.france24.com/en/rss",
        "el_pais": "https://feeds.elpais.com/mrss-s/pages/ep/site/elpais.com/portada",
        
        "guardian": "https://www.theguardian.com/world/rss",
        "nyt_world": "https://rss.nytimes.com/services/xml/rss/nyt/World.xml",
        "washington_post_world": "https://feeds.washingtonpost.com/rss/world",
         "npr_world":      "https://feeds.npr.org/1004/rss.xml",
        

        "ft": "https://www.ft.com/?format=rss",
        "bloomberg": "https://feeds.bloomberg.com/markets/news.rss",
        "forbes": "https://www.forbes.com/business/feed/",
         "market_watch":   "https://feeds.marketwatch.com/marketwatch/topstories/",
    "investing_com":  "https://www.investing.com/rss/news.rss",
    "yahoo_finance":  "https://finance.yahoo.com/news/rssindex",
          
}
       


for name, url in feeds.items():
    feed = feedparser.parse(url)
    entry_count = len(feed.entries)
    bozo = feed.bozo
    status = getattr(feed, "status", "N/A")
    
    if entry_count == 0 or bozo:
        print(f"❌ {name} | status:{status} | entries:{entry_count} | bozo:{bozo}")
    else:
        print(f"✅ {name} | status:{status} | entries:{entry_count}")

check mcp tool out

In [30]:
from langchain_mcp_adapters.client import MultiServerMCPClient
async def get_tools():
    client=MultiServerMCPClient(
        {"server":{
             "url":"http://localhost:8000/mcp",
            "transport":"streamable_http"
        }}     
    )
    tools=await client.get_tools()
    return tools

tools = await get_tools()
for tool in tools:
    print(tool.name)        
    print(tool.description) 
    print(tool.args_schema)
    print("---")

tavily_serach
Use this tool to search anything from internet like google serach.
     Fact-check or verify a specific claim using live web search. 
     Use this AFTER retrieving news from RSS tools to confirm accuracy.
{'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'tavily_serachArguments', 'type': 'object'}
---
wiki_search
Search query on wikipedia and return the summary
{'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'wiki_searchArguments', 'type': 'object'}
---
weather
Get Weather for the city
{'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'weatherArguments', 'type': 'object'}
---
get_old_news
Get old news from past using newsapi.org
{'properties': {'query': {'title': 'Query', 'type': 'string'}, 'days_ago': {'title': 'Days Ago', 'type': 'integer'}}, 'required': ['query', 'days_ago'], 'title': 'get_old_newsArguments', 'type': 'object'}
---
sp

In [32]:
import json
def convert_to_bedrock_tools(tools:list)->list:
    "Convert mcp tools to bedrock tools"
    bedrock_tools=[]
    for tool in tools:
        bedrock_tools.append({
            "toolSpec":{
                "name":tool.name,
            "description":tool.description,
            "inputSchema":{
                "json":{
                    "type":tool.args_schema["type"],
                    "properties":tool.args_schema["properties"],
                    "required":tool.args_schema["required"],
                    
                }
            }

            }
            
        })
    return bedrock_tools
tools= await get_tools()
result=convert_to_bedrock_tools(tools)
print(result)



[{'toolSpec': {'name': 'tavily_serach', 'description': 'Use this tool to search anything from internet like google serach.\n     Fact-check or verify a specific claim using live web search. \n     Use this AFTER retrieving news from RSS tools to confirm accuracy.', 'inputSchema': {'json': {'type': 'object', 'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query']}}}}, {'toolSpec': {'name': 'wiki_search', 'description': 'Search query on wikipedia and return the summary', 'inputSchema': {'json': {'type': 'object', 'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query']}}}}, {'toolSpec': {'name': 'weather', 'description': 'Get Weather for the city', 'inputSchema': {'json': {'type': 'object', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city']}}}}, {'toolSpec': {'name': 'get_old_news', 'description': 'Get old news from past using newsapi.org', 'inputSchema': {'json': {'type': 'object', 'properties': {'qu

In [53]:
langchain_tools= await get_tools()
bedrock_tools=convert_to_bedrock_tools(langchain_tools)
body={ 
"modelId": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
"messages": [{"role": "user", "content": [{"text": "What is current news in cricket"}]}],
"system": [{"text": "You are a research agent. Use the tools to fetch news."}],
"toolConfig": {"tools": bedrock_tools}
}
import boto3
client=boto3.client("bedrock-runtime",region_name="us-east-1")
response=client.converse(**body)

print("stopReason",response["stopReason"])
print("content",response["output"]["message"]["content"])
tool_block=None
for block in response["output"]["message"]["content"]:
    if "toolUse" in block:
        tool_block=block["toolUse"]
        break
print("tool name:", tool_block["name"])        
print("tool input:", tool_block["input"])     
print("tool use id:", tool_block["toolUseId"]) 

stopReason tool_use
content [{'text': "I'll fetch the latest cricket news for you."}, {'toolUse': {'toolUseId': 'tooluse_ap1ncsbqEg2zTEuzMbkeL2', 'name': 'sports_news', 'input': {'query': 'cricket news'}, 'type': 'tool_use'}}]
tool name: sports_news
tool input: {'query': 'cricket news'}
tool use id: tooluse_ap1ncsbqEg2zTEuzMbkeL2


In [61]:
tool_name=tool_block["name"]
tool_input=tool_block["input"]

matching_tool=None
for tool in langchain_tools:
    if tool.name==tool_name:
        matching_tool=tool
        break
print("found tool:",matching_tool.name)
result= await matching_tool.ainvoke(tool_input)
print("tool result",str(result)[:500])



found tool: sports_news
tool result [{'type': 'text', 'text': '[{\'title\': \'Stokes & Atkinson investigated over nightclub incident\', \'description\': \'An incident at a nightclub involving England cricket captain Ben Stokes, pace bowler Gus Atkinson and a Saracens academy player is being investigated by the England and Wales Cricket Board and the Prem Rugby club.\', \'link\': \'https://www.bbc.com/sport/cricket/articles/c0ry801k2geo?at_medium=RSS&at_campaign=rss\', \'source\': \'bbc_sport\', \'score\': 1}, {\'title\': \'Lewis H


/var/folders/f5/vlxr5wtj7d7fsc9qtq895d1w0000gn/T/ipykernel_6683/100165567.py:10: RuntimeWarning: coroutine 'StructuredTool.ainvoke' was never awaited
  result= await matching_tool.ainvoke(tool_input)


In [62]:
# build the messages with the full conversation so far
messages = [
    # original user message
    {"role": "user", "content": [{"text": "What is current news in cricket"}]},
    # assistant's tool call response
    {"role": "assistant", "content": response["output"]["message"]["content"]},
    # tool result we just got
    {"role": "user", "content": [
        {
            "toolResult": {
                "toolUseId": tool_block["toolUseId"],
                "content": [{"text": str(result)}]
            }
        }
    ]}
]

# send back to Bedrock
response2 = client.converse(
    modelId="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    messages=messages,
    system=[{"text": "You are a research agent. Use the tools to fetch news."}],
    toolConfig={"tools": bedrock_tools}
)

print("stopReason:", response2["stopReason"])
print("final text:", response2["output"]["message"]["content"][0]["text"][:500])

stopReason: end_turn
final text: Based on the latest sports news results, here's the current cricket news:

## Cricket News

**Top Cricket Story:**

🏏 **Ben Stokes & Gus Atkinson Investigation**
- England cricket captain **Ben Stokes** and pace bowler **Gus Atkinson** are being investigated over an incident that occurred at a nightclub
- The incident also involved a Saracens academy player
- The investigation is being conducted by the England and Wales Cricket Board (ECB) and Premiership Rugby club

---

*Note: The sports feed 


In [63]:
from dotenv import load_dotenv
load_dotenv()
import requests,os
def weather(city:str)->str:
    "Get Weather for the city"
    api_key=os.getenv("WEATHER_API_KEY")
    if not api_key:
        raise ValueError("NO apikey found in the environment")
    params=({
        "q":city,
        "appid":api_key,
        "units":"metric"
    })
    search=requests.get("http://api.openweathermap.org/data/2.5/weather",params=params)
    data=search.json()
    if data.get("cod") != 200:
        return "City not found"
    return str(data)

res=weather("chennai")
print(res)

{'coord': {'lon': 80.2785, 'lat': 13.0878}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 37.57, 'feels_like': 44.57, 'temp_min': 37.22, 'temp_max': 37.91, 'pressure': 1004, 'humidity': 47, 'sea_level': 1004, 'grnd_level': 1003}, 'visibility': 10000, 'wind': {'speed': 0.45, 'deg': 109, 'gust': 0.89}, 'clouds': {'all': 100}, 'dt': 1781083333, 'sys': {'type': 2, 'id': 2104103, 'country': 'IN', 'sunrise': 1781050320, 'sunset': 1781096684}, 'timezone': 19800, 'id': 1264527, 'name': 'Chennai', 'cod': 200}
